In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
os.environ["LANGSMITH_TRACING"] = "true"
openai_api_key = os.environ["OPENAI_API_KEY"]
langsmith_api_key=os.environ["LANGSMITH_API_KEY"]
print(f"openai_api_key: f{openai_api_key}")
print(f"langsmith_api_key: f{langsmith_api_key}")

openai_api_key: fsk-proj-9Jxxdmlda78k3iW4QJlGLyEfBOYuuKDKHdl2UpcwfayjeuCiNrbSacYCr7_V9CqN6eidA73dLLT3BlbkFJE-cMRhNA66194BKENB9gLQ9yrl5uNzLXGP0FGxsWoZ2zYmHjv7csOPSzc4ace0Rga2ZwJBHNUA
langsmith_api_key: flsv2_pt_cd916e91d2d1405f9b127950a731ca8e_98d207863e


In [5]:
### create the datapoints
from langsmith import Client
client = Client() # responsible for uploading the dataset
###
dataset_name = "@saranshkhulbe7/chatbot-evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {
                "answer": "A platform for observing and evaluating LLM applications"
            },
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
    ],
)


{'example_ids': ['88b5f4c4-da1e-440d-b251-5df53d9af043',
  'bf1dc485-e8b4-4617-8095-4bce31b0de8a',
  'de0cce0d-b4ec-487f-8934-17d4ea68dacf',
  '64faa0f3-ace5-4b52-8c6b-994de435ca17',
  'd634a02d-747a-4915-ad53-fd6e795c2850'],
 'count': 5,
 'as_of': '2026-08-16T13:55:02.025012171Z'}

In [ ]:
### Define Metrics (LLM As A Judge)

import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())
eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs["question"]}
    Here is the real answer:
    {reference_outputs["answer"]}
    You are grading the following predicted answer:
    {outputs["response"]}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "system", "content": eval_instructions}, {"role": "user", "content": user_content }]
    ).choices[0].message.content
    
    return response=="CORRECT"